In [5]:
import pandas as pd
import openpyxl

In [ ]:
path_population = "https://minio.lab.sspcloud.fr/mangane/Projet_PDS_2026/raw/TD_POP1A_2022.xlsx"
excel_file = pd.ExcelFile(path_population)
print(excel_file.sheet_names)

['COM', 'ARM', 'Liste des variables', 'Documentation']


In [ ]:
variables = pd.read_excel(
    path_population,
    sheet_name="Liste des variables"
)

print(variables.head(20))
print(variables.columns)

   Tableaux détaillés - Population par sexe et âge regroupé
0                                 Liste des variables      
1   Mise en ligne le 26/06/2025       Géographie a...      
2   ©Insee       Source(s) : Insee, RP2022 exploit...      
3                                                 NaN      
4                                 Variables utilisées      
5                                                 NaN      
6         AGEPYR10 : Âge regroupé (10 classes d'âges)      
7                                 00 : Moins de 3 ans      
8                                      03 : 3 à 5 ans      
9                                     06 : 6 à 10 ans      
10                                   11 : 11 à 17 ans      
11                                   18 : 18 à 24 ans      
12                                   25 : 25 à 39 ans      
13                                   40 : 40 à 54 ans      
14                                   55 : 55 à 64 ans      
15                                   65 

In [10]:
df = pd.read_excel(
    path_population,
    sheet_name="COM",
    header=10,
    dtype={"CODGEO": str}
)
df.head()

,CODGEO,LIBGEO,SEXE1_AGEPYR1000,SEXE1_AGEPYR1003,SEXE1_AGEPYR1006,SEXE1_AGEPYR1011,SEXE1_AGEPYR1018,SEXE1_AGEPYR1025,SEXE1_AGEPYR1040,SEXE1_AGEPYR1055,SEXE1_AGEPYR1065,SEXE1_AGEPYR1080,SEXE2_AGEPYR1000,SEXE2_AGEPYR1003,SEXE2_AGEPYR1006,SEXE2_AGEPYR1011,SEXE2_AGEPYR1018,SEXE2_AGEPYR1025,SEXE2_AGEPYR1040,SEXE2_AGEPYR1055,SEXE2_AGEPYR1065,SEXE2_AGEPYR1080
0,01001,L'Abergement-Clémenciat,19.000000,15.000000,36.000000,42.000000,15.000000,77.000000,87.000000,66.000000,60.000000,20.000000,14.000000,13.000000,20.000000,39.000000,15.000000,74.000000,90.000000,65.000000,68.000000,24.000000
1,01002,L'Abergement-de-Varey,3.000000,6.000000,7.000000,22.000000,6.000000,20.000000,39.000000,12.000000,18.000000,10.000000,3.000000,6.000000,12.000000,12.000000,5.000000,21.000000,35.000000,14.000000,14.000000,8.000000
2,01004,Ambérieu-en-Bugey,291.774098,327.193219,525.370992,719.637562,733.735172,1581.246385,1344.792069,833.964093,888.674859,244.547759,323.969423,283.369968,467.591432,664.536778,736.655842,1518.093099,1408.495207,984.296766,1151.504227,524.551050
3,01005,Ambérieux-en-Dombes,36.262404,37.108922,65.943276,80.082047,54.445778,193.448597,196.440979,137.738813,121.682540,36.336950,29.111892,33.799528,61.124887,81.584022,53.278755,195.596675,204.339272,131.092362,117.451751,50.130548
4,01006,Ambléon,2.000000,1.000000,2.000000,1.000000,4.000000,12.000000,11.000000,14.000000,11.000000,2.000000,1.000000,2.000000,2.000000,1.000000,2.000000,10.000000,7.000000,10.000000,15.000000,4.000000


In [11]:
# Colonnes contenant les effectifs de population
population_cols = [
    col for col in df.columns
    if col.startswith("SEXE")
]

# Population totale par commune
df["population"] = df[population_cols].sum(axis=1)

# Colonnes correspondant aux personnes de 65 ans ou plus
cols_65_plus = [
    "SEXE1_AGEPYR1065",
    "SEXE1_AGEPYR1080",
    "SEXE2_AGEPYR1065",
    "SEXE2_AGEPYR1080"
]

# Population de 65 ans ou plus
df["population_65_plus"] = df[cols_65_plus].sum(axis=1)

# Part des 65 ans ou plus
df["part_65_plus"] = df["population_65_plus"] / df["population"]

print(
    df[
        [
            "CODGEO",
            "LIBGEO",
            "population",
            "population_65_plus",
            "part_65_plus"
        ]
    ].head()
)

  CODGEO                   LIBGEO  population  population_65_plus  \
0  01001  L'Abergement-Clémenciat       859.0          172.000000   
1  01002    L'Abergement-de-Varey       273.0           50.000000   
2  01004        Ambérieu-en-Bugey     15554.0         2809.277895   
3  01005      Ambérieux-en-Dombes      1917.0          325.601789   
4  01006                  Ambléon       114.0           32.000000   

   part_65_plus  
0      0.200233  
1      0.183150  
2      0.180614  
3      0.169850  
4      0.280702  


In [12]:
# On va garder uniquement les variables utiles pour la suite
df_population = df[
    [
        "CODGEO",
        "LIBGEO",
        "population",
        "population_65_plus",
        "part_65_plus"
    ]
].copy()

df_population["part_65_plus_pct"] = (
    df_population["part_65_plus"] * 100
)

print(df_population.head())
print(df_population.shape)

  CODGEO                   LIBGEO  population  population_65_plus  \
0  01001  L'Abergement-Clémenciat       859.0          172.000000   
1  01002    L'Abergement-de-Varey       273.0           50.000000   
2  01004        Ambérieu-en-Bugey     15554.0         2809.277895   
3  01005      Ambérieux-en-Dombes      1917.0          325.601789   
4  01006                  Ambléon       114.0           32.000000   

   part_65_plus  part_65_plus_pct  
0      0.200233         20.023283  
1      0.183150         18.315018  
2      0.180614         18.061450  
3      0.169850         16.984966  
4      0.280702         28.070175  
(34858, 6)


In [ ]:
# Codes communes manquants
print(df_population["CODGEO"].isna().sum()) #non

# Doublons de codes communes
print(df_population["CODGEO"].duplicated().sum()) #non

0
0


In [14]:
path_population_clean = (
    "s3://mangane/Projet_PDS_2026/processed/population_communes_2022.csv"
)

df_population.to_csv(
    path_population_clean,
    index=False
)